In [1]:
from pathlib import Path

import io
import onnx
import onnxruntime.training.onnxblock as onnxblock
import torch
import timm
from onnxsim import simplify

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda-12.3'


In [2]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [3]:
# timm.list_models()

In [4]:
# Create a classifier instance
device = "cpu"
batch_size = 64
num_classes = 10
channels = 1
img_h, img_w = 28, 28

model_name = 'resnet18'

# artifacts path
artifacts_path = Path(f'./artifacts/{model_name}')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = timm.create_model(model_name, pretrained=False, num_classes=num_classes, in_chans=channels)#, img_size=32)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

# for param in pt_model.named_parameters():
#     if 'fc' not in param[0]:
#         param[1].requires_grad = False
#     # if 'bn' in param[0] or 'norm' in param[0]:
#     #     param[1].requires_grad = False
#     # print(param[0].ljust(30), '->\t', param[1].requires_grad)

# torch.save(pt_model, artifacts_path / f'{model_name}.pth')
# pt_model = torch.load(artifacts_path / f'{model_name}.pth')
# for param in pt_model.named_parameters():
#     if param[1].requires_grad:
#         print(param[0].ljust(40), '->\t', param[1].requires_grad)

# print(f'Model trainable parameters: {count_parameters(pt_model)}')

Model trainable parameters: 20320


### Generating forward-only graph

In [5]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device=device),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    training=torch.onnx.TrainingMode.TRAINING,
    dynamic_axes=dynamic_axes,
    export_params=True,
    keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')
# onnx_model, check = simplify(onnx_model)
# assert check, "Simplified ONNX model could not be validated"

### Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [6]:
# from onnxruntime.training import artifacts


# # artifacts path
# artifacts_path = Path(f'./artifacts/{model_name}')
# artifacts_path.mkdir(parents=True, exist_ok=True)

# requires_grad = []
# frozen_params = []
# for param in onnx_model.graph.initializer:
#     if 'bn' in param.name:
#         frozen_params.append(param.name)
#     else:
#         requires_grad.append(param.name)

# artifacts.generate_artifacts(
#     onnx_model,
#     requires_grad=requires_grad,
#     frozen_params=frozen_params,
#     loss=artifacts.LossType.CrossEntropyLoss,
#     optimizer=artifacts.OptimType.AdamW,
#     artifact_directory=artifacts_path,
# )

In [7]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [8]:
# Freeze layers that contains running_mean or running_var
frozen_layers = set(['.'.join(param.name.split('.')[:-1]) for param in onnx_model.graph.initializer if 'running_mean' in param.name or 'running_var' in param.name])

# Build the onnx model with loss
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    # if 'running_mean' in param.name or 'running_var' in param.name:
    layer_name = '.'.join(param.name.split('.')[:-1])
    if layer_name in frozen_layers:
        training_block.requires_grad(param.name, False)
    else:
        print(param.name)
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

2024-05-02 21:30:45,981 root [DEBUG] - Building training block CustomTrainingBlock
2024-05-02 21:30:45,982 root [DEBUG] - Building block: CrossEntropyLoss
2024-05-02 21:30:46,011 root [DEBUG] - Building gradient graph for training block CustomTrainingBlock
2024-05-02 21:30:46,015 root [DEBUG] - The loss output is onnx::loss::2. The gradient graph will be built starting from onnx::loss::2_grad.
2024-05-02 21:30:46,019 root [DEBUG] - Adding gradient accumulation nodes for training block CustomTrainingBlock
2024-05-02 21:30:46,022 root [DEBUG] - Building forward block AdamW
2024-05-02 21:30:46,025 root [DEBUG] - Building block: AdamWOptimizer


fc1.weight
fc1.bias
Graph output nodes: ('onnx::loss::2', 'output')


In [9]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')

Artifacts saved in artifacts/fc_net directory
